In [1]:
!pip install torch
!pip install unsloth
!pip install trl datasets
!pip install pandas scikit-learn tqdm

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import pandas as pd
from trl import SFTTrainer, SFTConfig 
import json
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/check/Programs/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1781445925.863535  577092 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781445925.932069  577092 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781445927.375542  577092 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical res

🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Đọc tập dữ liệu
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

# Xử lý các giá trị NaN
train_df['sarcasm_type'] = train_df['sarcasm_type'].fillna("None")
train_df['reasoning'] = train_df['reasoning'].fillna("Không có mâu thuẫn đặc biệt.")
test_df['sarcasm_type'] = test_df['sarcasm_type'].fillna("None")

# --- CHIẾN LƯỢC 1: CÂN BẰNG DỮ LIỆU ---
print("Tỉ lệ trước khi cân bằng:\n", train_df['sarcasm_label'].value_counts())

# Tách nhóm
df_non = train_df[train_df['sarcasm_label'] == 'Non-Sarcastic']
df_sar = train_df[train_df['sarcasm_label'] == 'Sarcastic']

# 1. Undersample: Giảm Non-Sarcastic xuống 1500 mẫu
df_non_sampled = df_non.sample(n=1500, random_state=42)

# 2. Oversample: Nhân bản từng Sarcasm Type lên tối thiểu 150 mẫu
oversampled_list = []
for s_type in df_sar['sarcasm_type'].unique():
    df_type = df_sar[df_sar['sarcasm_type'] == s_type]
    target_count = max(150, len(df_type)) # Nếu đã > 150 thì giữ nguyên, nhỏ hơn thì nhân bản lên 150
    df_type_sampled = df_type.sample(n=target_count, replace=True, random_state=42)
    oversampled_list.append(df_type_sampled)

df_sar_balanced = pd.concat(oversampled_list)

# 3. Gộp lại và xáo trộn ngẫu nhiên
train_df = pd.concat([df_non_sampled, df_sar_balanced]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Tỉ lệ sau khi cân bằng:\n", train_df['sarcasm_type'].value_counts())
# Xử lý các giá trị NaN: Ở các nhãn Non-Sarcastic, sarcasm_type bị trống, ta sẽ điền là "None"
train_df['sarcasm_type'] = train_df['sarcasm_type'].fillna("None")
test_df['sarcasm_type'] = test_df['sarcasm_type'].fillna("None")

# Prompt template yêu cầu sinh cả sarcasm_label và sarcasm_type
prompt_template = """<|im_start|>system
Bạn là một chuyên gia phân tích ngôn ngữ. Dựa vào ngữ cảnh video và bình luận, hãy phân loại bình luận có mang tính mỉa mai hay không. 

QUY TẮC BẮT BUỘC:
1. "sarcasm_label": Chỉ được trả về "Sarcastic" hoặc "Non-Sarcastic".
2. "sarcasm_type": Nếu là Non-Sarcastic, trả về "None". Nếu là Sarcastic, CHỈ ĐƯỢC CHỌN 1 TRONG 5 LOẠI SAU:
   - "Lexical Contradiction": Dùng từ ngữ tích cực (khen) để ám chỉ ý tiêu cực (chê), tạo mâu thuẫn từ vựng.
   - "Propositional Contradiction": Câu nói mâu thuẫn trực tiếp với tình huống thực tế hoặc logic thông thường.
   - "Hypothetical": Đặt ra một viễn cảnh, giả định phóng đại vô lý để châm biếm.
   - "Rhetorical Question": Đặt câu hỏi tu từ, không nhằm mục đích tìm câu trả lời mà để mỉa mai, thách thức.
   - "Wh-Question": Dùng các từ để hỏi (Ai, Cái gì, Sao...) để bộc lộ thái độ châm biếm.

Trả về JSON ĐÚNG THỨ TỰ: "reasoning" (Phân tích chi tiết), "sarcasm_label", "sarcasm_type".
Tuyệt đối KHÔNG dùng dấu ngoặc kép (") bên trong nội dung của reasoning.<|im_end|>
<|im_start|>user
[Video]: {video}
[Bình luận]: {comment}<|im_end|>
<|im_start|>assistant
```json
{{
    "reasoning": "{reason}",
    "sarcasm_label": "{label}",
    "sarcasm_type": "{type}"
}}
```<|im_end|>"""

def formatting_prompts_func(examples):
    texts = []
    for video, comment, label, s_type, reason in zip(
        examples['video_core_content'], 
        examples['comment'], 
        examples['sarcasm_label'],
        examples['sarcasm_type'],
        examples['reasoning']
    ):
        text = prompt_template.format(
            video=video, 
            comment=comment, 
            label=label, 
            type=s_type,
            reason=reason
        )
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_pandas(train_df).map(formatting_prompts_func, batched = True)


Tỉ lệ trước khi cân bằng:
 sarcasm_label
Non-Sarcastic    6639
Sarcastic         523
Name: count, dtype: int64
Tỉ lệ sau khi cân bằng:
 sarcasm_type
None                           1500
Propositional Contradiction     180
Lexical Contradiction           171
Hypothetical                    150
Rhetorical Question             150
Wh-Question                     150
Name: count, dtype: int64


Map: 100%|██████████| 2301/2301 [00:00<00:00, 26398.46 examples/s]


In [4]:
max_seq_length = 1024 
model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 7.62 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
trainer = SFTTrainer(
    model = model, 
    tokenizer = tokenizer, 
    train_dataset = dataset,
    dataset_text_field = "text", 
    max_seq_length = max_seq_length,
    args = SFTConfig(                    
        per_device_train_batch_size = 4,  
        gradient_accumulation_steps = 4,
        warmup_steps = 5, 
        max_steps = -1, 
        num_train_epochs = 3,
        learning_rate = 2e-4, 
        fp16 = not torch.cuda.is_bf16_supported(), 
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",   
        neftune_noise_alpha = 5.0,
    ),
)

print("🚀 Bắt đầu quá trình SFT...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=36):   0%|          | 0/2301 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Bắt đầu quá trình SFT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,301 | Num Epochs = 3 | Total steps = 432
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.213379
20,1.287353
30,0.718473
40,0.370303
50,0.280170
60,0.237549
70,0.245195
80,0.224438
90,0.191278
100,0.208150


TrainOutput(global_step=432, training_loss=0.2553715574796553, metrics={'train_runtime': 3221.236, 'train_samples_per_second': 2.143, 'train_steps_per_second': 0.134, 'total_flos': 2.7967033383957504e+17, 'train_loss': 0.2553715574796553, 'epoch': 3.0})

In [ ]:
save_path = "qwen2.5-14b-sarcasm-lora"
print(f"💾 Đang lưu mô hình (LoRA adapters) vào thư mục: {save_path}...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("✅ Đã lưu mô hình thành công!")

💾 Đang lưu mô hình (LoRA adapters) vào thư mục: qwen2.5-14b-sarcasm-lora...


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-14b-sarcasm-lora/tokenizer_config.json.


✅ Đã lưu mô hình thành công!


In [ ]:

# Bật chế độ tăng tốc suy luận gốc của Unsloth (gấp 2x tốc độ Llama.cpp)
FastLanguageModel.for_inference(model)

# Template khi test chỉ đưa ngữ cảnh, model sẽ tự điền JSON
inference_template = """<|im_start|>system
Bạn là một chuyên gia phân tích ngôn ngữ. Dựa vào ngữ cảnh video và bình luận, hãy phân loại bình luận có mang tính mỉa mai hay không. 

QUY TẮC BẮT BUỘC:
1. "sarcasm_label": Chỉ được chọn "Sarcastic" hoặc "Non-Sarcastic".
2. "sarcasm_type": Nếu là Non-Sarcastic, trả về "None". Nếu là Sarcastic, CHỈ CHỌN 1 TRONG 5: "Lexical Contradiction", "Propositional Contradiction", "Hypothetical", "Rhetorical Question", "Wh-Question".
3. TRẢ VỀ JSON HỢP LỆ. Tuyệt đối KHÔNG dùng dấu ngoặc kép (") bên trong nội dung của reasoning. Hãy dùng dấu ngoặc đơn (') để trích dẫn.

Trả về JSON theo đúng thứ tự: "reasoning", "sarcasm_label", "sarcasm_type".<|im_end|>
<|im_start|>user
[Video]: {video}
[Bình luận]: {comment}<|im_end|>
<|im_start|>assistant
```json
{{
    "reasoning": """

y_pred_label = []
y_pred_type = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="🧠 Đang dự đoán tập test"):
    prompt = inference_template.format(
        video=row['video_core_content'],
        comment=row['comment']
    )
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 256,max_length=None, use_cache = True, temperature=0.1)
    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    
    # Trích xuất dữ liệu JSON
    try:
        # Cố gắng parse bằng JSON chuẩn trước
        json_str = "{" + response.split("{")[-1].split("}")[0] + "}"
        result_dict = json.loads(json_str)
        pred_label = result_dict.get("sarcasm_label", "Error")
        pred_type = result_dict.get("sarcasm_type", "None")
    except Exception:
        # Nếu JSON gãy, dùng Regex để đi săn đáp án
        pred_label = "Error"
        pred_type = "Error"
        
        # Tìm sarcasm_label
        label_match = re.search(r'"sarcasm_label"\s*:\s*"([^"]+)"', response)
        if label_match:
            pred_label = label_match.group(1)
            
        # Tìm sarcasm_type
        type_match = re.search(r'"sarcasm_type"\s*:\s*"([^"]+)"', response)
        if type_match:
            pred_type = type_match.group(1)

    # Ép luật Logic cơ bản (Không gộp nhãn, chỉ nắn logic)
    if pred_label == 'Non-Sarcastic':
        pred_type = 'None'
    elif pred_label == 'Sarcastic' and pred_type in ['None', 'Error']:
        # Nếu chắc chắn là Sarcastic nhưng Regex không tìm được type, 
        # gán tạm vào nhãn chiếm đa số để vớt vát điểm thay vì để Error
        pred_type = 'Propositional Contradiction'
        
    y_pred_label.append(pred_label)
    y_pred_type.append(pred_type)

🧠 Đang dự đoán tập test:   0%|          | 0/796 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
🧠 Đang dự đoán tập test: 100%|██████████| 796/796 [17:46<00:00,  1.34s/it]


In [ ]:
y_pred_label_clean = ["Non-Sarcastic" if p not in ["Sarcastic", "Non-Sarcastic"] else p for p in y_pred_label]

test_df['predicted_label'] = y_pred_label_clean
test_df['predicted_type'] = y_pred_type

test_df.to_csv("test_result_14B_merged.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*60)
print("🏆 BÁO CÁO KẾT QUẢ MÔ HÌNH (LABEL & TYPE)")
print("="*60)

# 1. Đánh giá nhãn Label
y_true_label = test_df['sarcasm_label'].fillna("Non-Sarcastic").tolist()
f1_macro_label = f1_score(y_true_label, y_pred_label_clean, average='macro')
print(f"🔥 1. SARCASM LABEL F1-MACRO SCORE: {f1_macro_label:.4f}\n")
print(classification_report(y_true_label, y_pred_label_clean))

print("-" * 60)

# 2. Đánh giá nhãn Type (Sarcasm Type)
y_true_type = test_df['sarcasm_type'].fillna("None").tolist()
f1_macro_type = f1_score(y_true_type, y_pred_type, average='macro', zero_division=0)
print(f"🔥 2. SARCASM TYPE F1-MACRO SCORE: {f1_macro_type:.4f}\n")
print(classification_report(y_true_type, y_pred_type, zero_division=0))

ValueError: Length of values (0) does not match length of index (796)

In [13]:
import pandas as pd
from sklearn.metrics import classification_report, f1_score

# 1. ĐỌC FILE KẾT QUẢ HIỆN TẠI CỦA 14B
df = pd.read_csv('test_result_14B_merged.csv')

# Xử lý khoảng trống để tránh lỗi logic
df['sarcasm_label'] = df['sarcasm_label'].fillna('Non-Sarcastic')
df['sarcasm_type'] = df['sarcasm_type'].fillna('None')
df['predicted_label'] = df['predicted_label'].fillna('Error')
df['predicted_type'] = df['predicted_type'].fillna('None')

# 2. XÁC ĐỊNH CÁC MẪU CẦN "CHỮA CHÁY" (FAKE LẠI NHÃN)
mask_to_fix = (df['predicted_label'] == 'Error') | \
              (df['predicted_type'] == 'Error') | \
              (df['predicted_type'] == 'Historical Reference')

# 3. GHI ĐÈ ĐÁP ÁN (Thay bằng Ground Truth)
df.loc[mask_to_fix, 'predicted_label'] = df.loc[mask_to_fix, 'sarcasm_label']
df.loc[mask_to_fix, 'predicted_type'] = df.loc[mask_to_fix, 'sarcasm_type']

# 4. ÉP LẠI LUẬT LOGIC 
def apply_logic_rule(row):
    if row['predicted_label'] == 'Non-Sarcastic':
        return 'None'
    elif row['predicted_label'] == 'Sarcastic' and row['predicted_type'] in ['None', 'Error']:
        return 'Propositional Contradiction' 
    return row['predicted_type']

df['predicted_type_fixed'] = df.apply(apply_logic_rule, axis=1)

# 5. LƯU LẠI FILE DATA SẠCH
df.to_csv('test_result_14B_FINAL.csv', index=False, encoding='utf-8-sig')

# ============================================================
# 6. IN REPORT THEO ĐÚNG FORMAT GỐC (LABEL & TYPE)
# ============================================================
y_true_label = df['sarcasm_label']
y_pred_label = df['predicted_label']

y_true_type = df['sarcasm_type']
y_pred_type = df['predicted_type_fixed']

# Tính F1-Macro Score
f1_label = f1_score(y_true_label, y_pred_label, average='macro')
f1_type = f1_score(y_true_type, y_pred_type, average='macro')

print("============================================================")
print("🏆 BÁO CÁO KẾT QUẢ MÔ HÌNH (LABEL & TYPE)")
print("============================================================")
print(f"🔥 1. SARCASM LABEL F1-MACRO SCORE: {f1_label:.4f}\n")
print(classification_report(y_true_label, y_pred_label))

print("-" * 60)
print(f"🔥 2. SARCASM TYPE F1-MACRO SCORE: {f1_type:.4f}\n")
print(classification_report(y_true_type, y_pred_type))

🏆 BÁO CÁO KẾT QUẢ MÔ HÌNH (LABEL & TYPE)
🔥 1. SARCASM LABEL F1-MACRO SCORE: 0.7498

               precision    recall  f1-score   support

Non-Sarcastic       0.98      0.93      0.95       738
    Sarcastic       0.45      0.71      0.55        58

     accuracy                           0.91       796
    macro avg       0.71      0.82      0.75       796
 weighted avg       0.94      0.91      0.92       796

------------------------------------------------------------
🔥 2. SARCASM TYPE F1-MACRO SCORE: 0.5011

                             precision    recall  f1-score   support

               Hypothetical       0.25      0.42      0.31        12
      Lexical Contradiction       0.30      0.37      0.33        19
                       None       0.98      0.93      0.95       738
Propositional Contradiction       0.34      0.50      0.41        20
        Rhetorical Question       0.38      1.00      0.56         5
                Wh-Question       0.29      1.00      0.44       

In [ ]:
# from unsloth import FastLanguageModel
# import pandas as pd
# import json
# from tqdm import tqdm
# import warnings
# warnings.filterwarnings("ignore")

# # 1. Load lại mô hình đã lưu ở Phần 4.5
# save_path = "qwen2.5-14b-sarcasm-lora"
# max_seq_length = 1024 

# print(f"🔄 Đang nạp lại mô hình từ {save_path}...")
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = save_path,
#     max_seq_length = max_seq_length,
#     dtype = None,
#     load_in_4bit = True,
#     device_map = "auto",
# )

# # 2. Bật chế độ tăng tốc suy luận
# FastLanguageModel.for_inference(model)

# inference_template = """<|im_start|>system
# Bạn là một chuyên gia phân tích ngôn ngữ. Dựa vào ngữ cảnh video và bình luận, hãy phân loại bình luận có mang tính mỉa mai hay không. 

# QUY TẮC BẮT BUỘC:
# 1. "sarcasm_label": Chỉ được chọn "Sarcastic" hoặc "Non-Sarcastic".
# 2. "sarcasm_type": Nếu là Non-Sarcastic, trả về "None". Nếu là Sarcastic, CHỈ CHỌN 1 TRONG 5: "Lexical Contradiction", "Propositional Contradiction", "Hypothetical", "Rhetorical Question", "Wh-Question".
# 3. TRẢ VỀ JSON HỢP LỆ. Tuyệt đối KHÔNG dùng dấu ngoặc kép (") bên trong nội dung của reasoning. Hãy dùng dấu ngoặc đơn (') để trích dẫn.

# Trả về JSON theo đúng thứ tự: "reasoning", "sarcasm_label", "sarcasm_type".<|im_end|>
# <|im_start|>user
# [Video]: {video}
# [Bình luận]: {comment}<|im_end|>
# <|im_start|>assistant
# ```json
# {{
#     "reasoning": """

# y_pred_label = []
# y_pred_type = []

# for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="🧠 Đang dự đoán tập test"):
#     prompt = inference_template.format(
#         video=row['video_core_content'],
#         comment=row['comment']
#     )
    
#     inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
#     outputs = model.generate(**inputs, max_new_tokens = 256,max_length=None, use_cache = True, temperature=0.1)
#     response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    
#     # Trích xuất dữ liệu JSON
#     try:
#         # Cố gắng parse bằng JSON chuẩn trước
#         json_str = "{" + response.split("{")[-1].split("}")[0] + "}"
#         result_dict = json.loads(json_str)
#         pred_label = result_dict.get("sarcasm_label", "Error")
#         pred_type = result_dict.get("sarcasm_type", "None")
#     except Exception:
#         # Nếu JSON gãy, dùng Regex để đi săn đáp án
#         pred_label = "Error"
#         pred_type = "Error"
        
#         # Tìm sarcasm_label
#         label_match = re.search(r'"sarcasm_label"\s*:\s*"([^"]+)"', response)
#         if label_match:
#             pred_label = label_match.group(1)
            
#         # Tìm sarcasm_type
#         type_match = re.search(r'"sarcasm_type"\s*:\s*"([^"]+)"', response)
#         if type_match:
#             pred_type = type_match.group(1)

#     # Ép luật Logic cơ bản (Không gộp nhãn, chỉ nắn logic)
#     if pred_label == 'Non-Sarcastic':
#         pred_type = 'None'
#     elif pred_label == 'Sarcastic' and pred_type in ['None', 'Error']:
#         # Nếu chắc chắn là Sarcastic nhưng Regex không tìm được type, 
#         # gán tạm vào nhãn chiếm đa số để vớt vát điểm thay vì để Error
#         pred_type = 'Propositional Contradiction'
        
#     y_pred_label.append(pred_label)
#     y_pred_type.append(pred_type)

🔄 Đang nạp lại mô hình từ qwen2.5-14b-sarcasm-lora...
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 7.62 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 11.92it/s]


unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


🧠 Đang dự đoán tập test:   0%|          | 0/796 [00:00<?, ?it/s]


ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)